In [2]:
import csv
from collections import defaultdict


def load_csv(path):
    """
    Open the CSV file and read all its data.
    Returns a tuple of (header, rows) where:
    - header is a list of column names
    - rows is a list of lists, each representing a data row
    """
    with open(path, newline="") as f:
        reader = csv.reader(f)

        # Read the first row separately because it contains the header
        header = next(reader)

        # Store only non-empty rows
        rows = [row for row in reader if row]

    return header, rows

def get_new_tuple_interactively(header, rows):
    """
    Ask the user to enter values for each attribute one by one.
    """
    attributes = header[:-1]  # all columns except the last one
    new_tuple = {}

    print("\nEnter the values for the tuple you want to classify:")
    print("-" * 60)

    for i, attr in enumerate(attributes):
        # Find all unique values available in this column
        possible_values = sorted(set(row[i] for row in rows))

        print(f"{attr} -> possible values: {possible_values}")

        # Take input from the user for this attribute
        value = input(f"Enter value for '{attr}': ").strip()

        # Save user input in the dictionary
        new_tuple[attr] = value

    return new_tuple


def naive_bayes(header, rows, new_tuple):
    attributes = header[:-1]
    class_col = header[-1]
    total_rows = len(rows)

    print("\n" + "=" * 60)
    print(f"Attributes: {attributes}")
    print(f"Class column: {class_col}")
    print("=" * 60)

    # --------------------------------------------------
    # STEP 1: Group all rows based on class label
    # --------------------------------------------------
    class_groups = defaultdict(list)

    for row in rows:
        # row[-1] is the class label
        class_groups[row[-1]].append(row)

    print("STEP 1: Count rows in each class")
    for label, group in class_groups.items():
        print(f"  {class_col} = {label}: {len(group)} rows")

    print("=" * 60)

    # --------------------------------------------------
    # STEP 2: Calculate prior probabilities
    # P(class) = number of rows in that class / total rows
    # --------------------------------------------------
    priors = {}

    for label, group in class_groups.items():
        priors[label] = len(group) / total_rows

    print("STEP 2: Prior probabilities")
    for label, prob in priors.items():
        count = len(class_groups[label])
        print(f"  P({class_col}={label}) = {count}/{total_rows} = {prob:.4f}")

    print("=" * 60)

    # --------------------------------------------------
    # STEP 3: Compute conditional probabilities
    # P(attribute=value | class)
    # --------------------------------------------------
    print("STEP 3: Conditional probabilities for the input tuple")
    print(f"Tuple X = {new_tuple}")
    print("-" * 60)

    # Store conditional probabilities for each class
    cond_probs = {label: [] for label in class_groups}

    for attr in attributes:
        # Find the column index of the current attribute
        col_index = header.index(attr)

        # Value entered by the user for this attribute
        value = new_tuple[attr]

        for label, group in class_groups.items():
            # Count how many rows in this class have the same value
            count = sum(1 for row in group if row[col_index] == value)
            total = len(group)

            # If count is 0, apply smoothing so probability does not become zero
            if count == 0:
                unique_values = len(set(row[col_index] for row in rows))
                prob = 1 / (total + unique_values)
                note = " (Laplace smoothed)"
            else:
                prob = count / total
                note = ""

            # Save the probability for later multiplication
            cond_probs[label].append(prob)

            print(
                f"  P({attr}={value} | {class_col}={label}) = "
                f"{count}/{total} = {prob:.4f}{note}"
            )

    print("=" * 60)

    # --------------------------------------------------
    # STEP 4: Multiply all conditional probabilities
    # Naive Bayes assumes all attributes are independent
    # --------------------------------------------------
    print("STEP 4: Multiply conditional probabilities")
    likelihoods = {}

    for label, probs in cond_probs.items():
        product = 1.0
        formula = []

        for p in probs:
            product *= p
            formula.append(f"{p:.4f}")

        likelihoods[label] = product
        print(f"  P(X | {class_col}={label}) = {' * '.join(formula)} = {product:.6f}")

    print("=" * 60)

    # --------------------------------------------------
    # STEP 5: Multiply by prior probabilities
    # This gives the Bayes numerator for each class
    # --------------------------------------------------
    print("STEP 5: Multiply by prior probabilities")
    scores = {}

    for label in class_groups:
        scores[label] = likelihoods[label] * priors[label]
        print(
            f"  P(X | {label}) * P({label}) = "
            f"{likelihoods[label]:.6f} * {priors[label]:.4f} = {scores[label]:.6f}"
        )

    print("=" * 60)

    # --------------------------------------------------
    # STEP 6: Normalize the scores
    # This converts scores into posterior probabilities
    # --------------------------------------------------
    total_score = sum(scores.values())
    print("STEP 6: Posterior probabilities")

    for label, score in scores.items():
        posterior = score / total_score if total_score != 0 else 0
        print(
            f"  P({class_col}={label} | X) = "
            f"{score:.6f}/{total_score:.6f} = {posterior:.4f}"
        )

    print("=" * 60)

    # --------------------------------------------------
    # STEP 7: Choose the class with the highest score
    # That class becomes the final prediction
    # --------------------------------------------------
    prediction = max(scores, key=scores.get)

    print(f"STEP 7: FINAL PREDICTION -> {class_col} = {prediction}")

    return prediction


if __name__ == "__main__":
    # Ask the user for the CSV file path
    csv_path = input("Enter path to your CSV file: ").strip()

    # Load the dataset
    header, rows = load_csv(csv_path)

    print(f"\nLoaded {len(rows)} rows from '{csv_path}'")

    # Ask the user to enter the values of the tuple to classify
    new_tuple = get_new_tuple_interactively(header, rows)

    # Run Naive Bayes classification
    naive_bayes(header, rows, new_tuple)


Loaded 303 rows from 'Heart.csv'

Enter the values for the tuple you want to classify:
------------------------------------------------------------
 -> possible values: ['1', '10', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '11', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '12', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '13', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '14', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '15', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '16', '160', '161', '162', '163', '164', '165', '166', '167', '168', '169', '17', '170', '171', '172', '173', '174', '175', '176', '177', '178', '179', '18', '180', '181', '182', '183', '184', '185', '186', '187', '188', '189', '19', '190', '191', '192', '193', '194', '195', '196', '197', '198', '199', '2', '20', '200', '201', '202', '203', '204', '205', '206', '207

In [ ]:
import csv
from collections import defaultdict, Counter


def load_csv(path):
    """
    Read the CSV file and return the header and data rows.
    Assumptions:
    - First row is the header
    - Last column is the class label
    - All other columns are categorical attributes
    """
    with open(path, newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        rows = [row for row in reader if row]
    return header, rows


def get_new_tuple_interactively(header, rows):
    """
    Ask the user to enter one value for each attribute.
    The last column is skipped because it is the class label.
    """
    attributes = header[:-1]
    new_tuple = {}

    print("\nEnter the values for the tuple you want to classify:")
    print("-" * 60)

    for i, attr in enumerate(attributes):
        possible_values = sorted(set(row[i] for row in rows))
        print(f"{attr} -> possible values: {possible_values}")
        value = input(f"Enter value for '{attr}': ").strip()
        new_tuple[attr] = value

    return new_tuple


def naive_bayes_fast(header, rows, new_tuple):
    """
    Naive Bayes classifier for categorical data.

    This version is faster because:
    - it stores class counts once,
    - it stores attribute-value counts once,
    - it avoids scanning the full dataset repeatedly.
    """
    attributes = header[:-1]
    class_col = header[-1]
    total_rows = len(rows)

    print("\n" + "=" * 60)
    print(f"Attributes: {attributes}")
    print(f"Class column: {class_col}")
    print("=" * 60)

    # --------------------------------------------------
    # STEP 1: Count how many rows belong to each class
    # --------------------------------------------------
    class_counts = Counter(row[-1] for row in rows)

    print("STEP 1: Class counts")
    for label, count in class_counts.items():
        print(f"  {class_col} = {label}: {count} rows")

    print("=" * 60)

    # --------------------------------------------------
    # STEP 2: Compute prior probabilities
    # --------------------------------------------------
    priors = {}
    print("STEP 2: Prior probabilities")
    for label, count in class_counts.items():
        priors[label] = count / total_rows
        print(f"  P({class_col}={label}) = {count}/{total_rows} = {priors[label]:.4f}")

    print("=" * 60)

    # --------------------------------------------------
    # STEP 3: Precompute counts for each attribute value
    # This avoids counting the same thing again and again
    # --------------------------------------------------
    print("STEP 3: Building frequency tables")
    attr_value_counts = {}

    for attr_index, attr in enumerate(attributes):
        # For each class, store how many times each value appears
        attr_value_counts[attr] = defaultdict(Counter)

        for row in rows:
            label = row[-1]
            value = row[attr_index]
            attr_value_counts[attr][label][value] += 1

    print("  Frequency tables ready.")
    print("=" * 60)

    # --------------------------------------------------
    # STEP 4: Compute conditional probabilities for the input tuple
    # --------------------------------------------------
    print("STEP 4: Conditional probabilities for the input tuple")
    print(f"Tuple X = {new_tuple}")
    print("-" * 60)

    class_scores = {}

    for label in class_counts:
        print(f"Class = {label}")
        score = priors[label]

        for attr_index, attr in enumerate(attributes):
            value = new_tuple[attr]
            count = attr_value_counts[attr][label][value]
            total_in_class = class_counts[label]

            if count == 0:
                # Simple Laplace smoothing
                num_possible_values = len(set(row[attr_index] for row in rows))
                prob = 1 / (total_in_class + num_possible_values)
                note = " (Laplace smoothed)"
            else:
                prob = count / total_in_class
                note = ""

            score *= prob

            print(
                f"  P({attr}={value} | {class_col}={label}) = "
                f"{count}/{total_in_class} = {prob:.4f}{note}"
            )

        class_scores[label] = score
        print(f"  Score for class {label} = {score:.6f}")
        print("-" * 60)

    # --------------------------------------------------
    # STEP 5: Normalize scores to get posterior probabilities
    # --------------------------------------------------
    total_score = sum(class_scores.values())

    print("STEP 5: Posterior probabilities")
    for label, score in class_scores.items():
        posterior = score / total_score if total_score != 0 else 0
        print(
            f"  P({class_col}={label} | X) = "
            f"{score:.6f}/{total_score:.6f} = {posterior:.4f}"
        )

    print("=" * 60)

    # --------------------------------------------------
    # STEP 6: Final prediction
    # --------------------------------------------------
    prediction = max(class_scores, key=class_scores.get)
    print(f"STEP 6: FINAL PREDICTION -> {class_col} = {prediction}")

    return prediction


if __name__ == "__main__":
    csv_path = input("Enter path to your CSV file: ").strip()
    header, rows = load_csv(csv_path)

    print(f"\nLoaded {len(rows)} rows from '{csv_path}'")
    new_tuple = get_new_tuple_interactively(header, rows)
    naive_bayes_fast(header, rows, new_tuple)


Loaded 303 rows from 'Heart.csv'

Enter the values for the tuple you want to classify:
------------------------------------------------------------
 -> possible values: ['1', '10', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '11', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '12', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '13', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '14', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '15', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '16', '160', '161', '162', '163', '164', '165', '166', '167', '168', '169', '17', '170', '171', '172', '173', '174', '175', '176', '177', '178', '179', '18', '180', '181', '182', '183', '184', '185', '186', '187', '188', '189', '19', '190', '191', '192', '193', '194', '195', '196', '197', '198', '199', '2', '20', '200', '201', '202', '203', '204', '205', '206', '207